**Convert pdf to text**

In [1]:
from pypdf import PdfReader

def extract_pdf(path):
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        txt = page.extract_text() or ""
        pages.append({"page": i+1, "text": txt})
    return pages

pages = extract_pdf("D:/Zecser/B110-Muhammed-Salman-Phase-4/Data/it_act_2000_updated.pdf")
len(pages)


36

**Chunking**

In [2]:
def chunk_pages(pages, chunk_size=350, overlap=80):
    chunks = []
    for p in pages:
        words = p["text"].split()
        i = 0
        while i < len(words):
            part = " ".join(words[i:i+chunk_size])
            chunks.append({
                "page": p["page"],
                "text": part
            })
            i += chunk_size - overlap
    return chunks

chunks = chunk_pages(pages)
len(chunks)


92

**Embeddings and FAISS index**

In [4]:
from sentence_transformers import SentenceTransformer
import faiss, json
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]
embs = model.encode(texts, convert_to_numpy=True)
faiss.normalize_L2(embs)

index = faiss.IndexFlatIP(embs.shape[1])
index.add(embs)

faiss.write_index(index, "D:/Zecser/B110-Muhammed-Salman-Phase-4/Data/faiss_index.bin")

# Save metadata
with open("D:/Zecser/B110-Muhammed-Salman-Phase-4/Data/metadata.jsonl", "w") as f:
    for i, c in enumerate(chunks):
        c["id"] = i
        f.write(json.dumps(c) + "\n")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Anaconda3\envs\intern\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Muhammed Salman\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]